# Lecture 4 · Neural Networks and Deep Learning — Exercises

Companion to the note [Lecture Notes 1-12](../Lecture%20Notes%201-12.md).

Practise the material of **Lecture 4** using **NumPy only** (no PyTorch): neurons and activation functions (sigmoid, tanh, ReLU and their
derivatives), why depth needs non-linearity, loss functions, **backpropagation** on computational graphs (the $x_1, w_1, x_2, w_2 \to y_6$ example of
the slides), softmax + cross-entropy, the **vanishing/exploding gradient** problems and gradient clipping, and stochastic gradient descent with
mini-batches, momentum and Nesterov. You will build and gradient-check a small multilayer perceptron from scratch, i.e. everything
`loss.backward()` and `optimizer.step()` do for you in the self-study.

**20 exercises** · 🧠 Concept ×5 · ✍️ By hand ×8 · 💻 Code ×7

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

import matplotlib.pyplot as plt
from scipy.special import expit, softmax, log_softmax
from sklearn.datasets import make_moons
rng = np.random.default_rng(4)

def num_grad(f, x, h=1e-6):
    # central finite differences of a scalar function f at the array x (x is modified in place and restored)
    g = np.zeros_like(x, dtype=float)
    it = np.nditer(x, flags=['multi_index'])
    for _ in it:
        i = it.multi_index
        old = x[i]
        x[i] = old + h; fp = f()
        x[i] = old - h; fm = f()
        x[i] = old
        g[i] = (fp - fm) / (2 * h)
    return g

## Part A · Concepts

### Exercise 1 · Why depth needs non-linearity
*🧠 Concept · ★☆☆*

(a) Show that a network with two linear layers and **no** activation, $\mathbf{y} = \mathbf{W}_2(\mathbf{W}_1\mathbf{x}+\mathbf{b}_1)+\mathbf{b}_2$, is a single affine map.
(b) Can such a network solve XOR? (c) Would a network with ReLU activations but where every pre-activation is always positive be any better?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Multiply out. For (c): what is ReLU on the positive half-line?

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $\mathbf{y} = (\mathbf{W}_2\mathbf{W}_1)\mathbf{x} + (\mathbf{W}_2\mathbf{b}_1+\mathbf{b}_2) = \mathbf{W}_{\text{eq}}\mathbf{x}+\mathbf{b}_{\text{eq}}$; any depth collapses this way.
(b) No: it is a linear model, and XOR is not linearly separable ([[Lecture Notes 1-12#Lecture 2: Linear Models for Classification II]]).
(c) No: on $z>0$ ReLU is the identity, so the network is again linear. Non-linearity only helps if units actually switch between regimes on the data.

</details>

### Exercise 2 · The self-study network has a one-unit bottleneck
*🧠 Concept · ★★☆*

The self-study builds `nn.Sequential(nn.Linear(2, 1), nn.ReLU(), nn.Linear(1, 1), nn.Sigmoid())` with `BCELoss`.
(a) Write the function it computes. (b) What shape can its decision boundary $\{\mathbf{x}: P(y=1\mid\mathbf{x}) = 0.5\}$ have? Is it more powerful than logistic regression?
(c) What would you change to learn the moons dataset?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

The hidden layer is a single number $h = \max(0, \mathbf{w}^T\mathbf{x}+b)$. The output is a monotone function of $h$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $P(y=1\mid\mathbf{x}) = \sigma\big(a\max(0, \mathbf{w}^T\mathbf{x}+b) + c\big)$.
(b) The output depends on $\mathbf{x}$ only through the scalar $\mathbf{w}^T\mathbf{x}+b$, monotonically. Its level sets are lines parallel to $\mathbf{w}^T\mathbf{x}+b=0$,
so the boundary is a **straight line** (or empty). It is no more expressive than logistic regression; worse, if the ReLU is inactive on all data the gradient is zero ("dead" unit).
(c) Widen the hidden layer (e.g. 16 units) so the output combines several half-planes into a curved boundary, as in Exercise 17.

</details>

### Exercise 3 · The training loop and the loss in PyTorch
*🧠 Concept · ★★☆*

(a) Put these in the right order and say what each does: `loss.backward()`, `optimizer.step()`, `optimizer.zero_grad()`, `predictions = model(inputs)`, `loss = criterion(...)`.
(b) What goes wrong if `zero_grad()` is forgotten?
(c) Why must a network trained with `nn.CrossEntropyLoss` **not** end with a softmax layer, and what numerical trick does the loss use?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

PyTorch *adds* new gradients into `.grad` buffers.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) `zero_grad()` (reset buffers), forward pass `model(inputs)`, `criterion(...)` (scalar loss), `backward()` (backprop: fills `.grad` with $\partial\mathcal L/\partial\theta$), `step()` (update, e.g. $\theta\leftarrow\theta-\eta\,\mathbf{g}$).
(b) Gradients accumulate over iterations: step $t$ uses $\sum_{s\le t}\mathbf{g}_s$, i.e. stale, growing and wrong updates.
(c) `CrossEntropyLoss` expects **logits** and applies `log_softmax` internally; adding a softmax would apply it twice. It computes
$\log\text{softmax}(\mathbf{z})_k = z_k - \log\sum_je^{z_j}$ with the **log-sum-exp** trick (subtract $\max_j z_j$), avoiding overflow and $\log 0$ (Exercise 16).

</details>

### Exercise 4 · Vanishing vs. exploding gradients
*🧠 Concept · ★★☆*

(a) Explain the "decimal chain reaction" for a deep sigmoid network. Why is tanh slightly better and ReLU much better?
(b) What causes exploding gradients, and how does **gradient clipping** help without changing the descent direction?
(c) Name two other remedies used in practice.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Backprop multiplies one local derivative per layer: $\prod_l f'(z^{(l)})$ (times weight matrices).

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Each sigmoid layer contributes a factor $\sigma'(z)\le0.25$, so the gradient reaching layer 1 shrinks like $0.25^L$ and the early layers stop learning.
$\tanh'$ is at most 1 (and zero-centred outputs), better but still saturates; ReLU has derivative exactly 1 for $z>0$, so the product does not decay (Exercise 18).
(b) Products of large Jacobians (large weights, "cliffs" in the loss surface) blow up; one huge step can throw the parameters far away. Clipping rescales $\mathbf{g}\leftarrow v\,\mathbf{g}/\lVert\mathbf{g}\rVert$
when $\lVert\mathbf{g}\rVert>v$: same direction, bounded length.
(c) Careful initialisation (Xavier/He), batch normalisation, residual connections, gated units (LSTM).

</details>

### Exercise 5 · Mini-batches and momentum
*🧠 Concept · ★★☆*

(a) Why do practitioners use batch sizes like 32 to 256 instead of the full dataset, given $SE = \sigma/\sqrt B$?
(b) Interpret $\frac1{1-\alpha}$ in momentum. Why does momentum help in a long narrow valley?
(c) What is the one difference between classical and Nesterov momentum?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Cost grows linearly with $B$, accuracy only like $\sqrt B$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Going from $B = 100$ to $10\,000$ costs $100\times$ more per step but reduces the noise only $10\times$. Many cheap noisy steps beat few exact ones, and the noise can even help escape poor regions.
(b) At terminal velocity the step is $\eta\lVert\mathbf{g}\rVert/(1-\alpha)$: $\alpha = 0.9$ means $10\times$ the plain step along directions where the gradient is consistent.
In a narrow valley the oscillating components across the valley cancel in the velocity while the consistent component along the valley accumulates (Exercise 19).
(c) Nesterov evaluates the gradient at the look-ahead point $\mathbf{w}_{t-1}+\alpha\mathbf{v}_{t-1}$ instead of at $\mathbf{w}_{t-1}$, correcting overshoot earlier.

</details>

## Part B · By hand

### Exercise 6 · Collapse a linear network
*✍️ By hand · ★☆☆*

Two linear layers (column convention $\mathbf{h} = \mathbf{W}_1\mathbf{x}+\mathbf{b}_1$, $\mathbf{y} = \mathbf{W}_2\mathbf{h}+\mathbf{b}_2$) with
$$\mathbf{W}_1 = \begin{pmatrix}1&2\\0&1\end{pmatrix},\ \mathbf{b}_1 = \begin{pmatrix}1\\-1\end{pmatrix},\quad \mathbf{W}_2 = \begin{pmatrix}3&0\\1&-1\end{pmatrix},\ \mathbf{b}_2 = \begin{pmatrix}0\\2\end{pmatrix}.$$
Compute the equivalent single layer $\mathbf{W}_{\text{eq}}, \mathbf{b}_{\text{eq}}$.

In [ ]:
W_eq = None
b_eq = None

_W1 = np.array([[1, 2], [0, 1.]]); _b1 = np.array([1, -1.]); _W2 = np.array([[3, 0], [1, -1.]]); _b2 = np.array([0, 2.])
_X = rng.normal(size=(5, 2)); _Y = ((_X @ _W1.T + _b1) @ _W2.T + _b2)
if W_eq is not None:
    check('same outputs as the 2-layer net', np.asarray(_X) @ np.asarray(W_eq).T + np.asarray(b_eq), _Y)
else:
    check('same outputs as the 2-layer net', None, 0)

<details>
<summary><b>💡 Hint</b></summary>

$\mathbf{W}_{\text{eq}} = \mathbf{W}_2\mathbf{W}_1$, $\mathbf{b}_{\text{eq}} = \mathbf{W}_2\mathbf{b}_1 + \mathbf{b}_2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\mathbf{W}_2\mathbf{W}_1 = \begin{pmatrix}3&6\\1&1\end{pmatrix}$, $\mathbf{W}_2\mathbf{b}_1 = (3, 2)^T$, so $\mathbf{b}_{\text{eq}} = (3, 4)^T$.
(The check compares outputs on random inputs, so any correct form passes.)

```python
W_eq = np.array([[3, 6], [1, 1.]])
b_eq = np.array([3, 4.])
```

</details>

### Exercise 7 · Forward pass of the slide graph
*✍️ By hand · ★☆☆*

$x_1 = 1, w_1 = 2, x_2 = 3, w_2 = 0.5$, target $o = 1$:
$y_1 = x_1w_1$, $y_2 = x_2w_2$, $y_3 = y_1+y_2$, $y_4 = \sigma(y_3)$, $y_5 = \text{ReLU}(y_3)$, $y_6 = y_4y_5$, $\text{err} = (o-y_6)^2$.
Compute every node (4 decimals).

In [ ]:
y3 = None
y4 = None
y6 = None
err = None

_y3 = 1*2 + 3*0.5
check('y3', y3, _y3); check('y4', y4, expit(_y3), tol=1e-4)
check('y6', y6, expit(_y3) * max(_y3, 0), tol=1e-4); check('err', err, (1 - expit(_y3) * _y3) ** 2, tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

$\sigma(3.5) = 1/(1+e^{-3.5})$, and $e^{-3.5}\approx0.0302$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$y_1 = 2$, $y_2 = 1.5$, $y_3 = 3.5$, $y_4 = \sigma(3.5)\approx0.970688$, $y_5 = 3.5$, $y_6\approx3.397406$, $\text{err} = (1-3.397406)^2\approx\mathbf{5.74756}$.

```python
y3 = 3.5
y4 = 1 / (1 + np.exp(-3.5))
y6 = y4 * 3.5
err = (1 - y6) ** 2
```

</details>

### Exercise 8 · Backward pass of the slide graph
*✍️ By hand · ★★☆*

Continue the previous exercise: compute $\partial\text{err}/\partial y_6$, $\partial\text{err}/\partial y_3$ (two branches meet here!), and finally
$\partial\text{err}/\partial w_1$, $\partial\text{err}/\partial w_2$, $\partial\text{err}/\partial x_1$, $\partial\text{err}/\partial x_2$.

In [ ]:
d_y3 = None
d_w1 = None
d_w2 = None
d_x1 = None
d_x2 = None

def _err(x1, w1, x2, w2):
    a = x1 * w1 + x2 * w2
    return (1 - expit(a) * max(a, 0)) ** 2
_h = 1e-6; _p = [1., 2., 3., 0.5]
_g = [(_err(*[v + _h * (i == k) for i, v in enumerate(_p)]) - _err(*[v - _h * (i == k) for i, v in enumerate(_p)])) / (2 * _h) for k in range(4)]
check('d err/d y3', d_y3, _g[1] / _p[0], tol=1e-3)
check('d err/d w1', d_w1, _g[1], tol=1e-3); check('d err/d w2', d_w2, _g[3], tol=1e-3)
check('d err/d x1', d_x1, _g[0], tol=1e-3); check('d err/d x2', d_x2, _g[2], tol=1e-3)

<details>
<summary><b>💡 Hint 1</b></summary>

$\partial\text{err}/\partial y_6 = -2(o-y_6)$; then $\partial/\partial y_4 = (\cdot)\,y_5$ and $\partial/\partial y_5 = (\cdot)\,y_4$ (product rule).

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

At $y_3$ **sum** the two paths: $\frac{\partial\text{err}}{\partial y_4}\sigma'(y_3) + \frac{\partial\text{err}}{\partial y_5}\cdot1$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\partial\text{err}/\partial y_6 = -2(1-3.397406) = 4.794812$.
$\partial/\partial y_4 = 4.794812\cdot3.5 = 16.78184$; $\partial/\partial y_5 = 4.794812\cdot0.970688 = 4.654268$.
$\sigma'(3.5) = 0.970688\cdot0.029312 = 0.028453$.
$\partial\text{err}/\partial y_3 = 16.78184\cdot0.028453 + 4.654268 = 0.47750 + 4.65427 = \mathbf{5.1318}$ (the slide's 5.132).
Add node: $\partial/\partial y_1 = \partial/\partial y_2 = 5.1318$.
$\partial\text{err}/\partial w_1 = 5.1318\cdot x_1 = \mathbf{5.132}$, $\partial\text{err}/\partial w_2 = 5.1318\cdot x_2 = \mathbf{15.395}$,
$\partial\text{err}/\partial x_1 = 5.1318\cdot w_1 = \mathbf{10.264}$, $\partial\text{err}/\partial x_2 = 5.1318\cdot w_2 = \mathbf{2.566}$.

```python
_s = 1 / (1 + np.exp(-3.5)); _y6 = _s * 3.5
_d6 = -2 * (1 - _y6)
d_y3 = _d6 * 3.5 * _s * (1 - _s) + _d6 * _s * 1
d_w1, d_w2, d_x1, d_x2 = d_y3 * 1, d_y3 * 3, d_y3 * 2, d_y3 * 0.5
```

</details>

### Exercise 9 · How fast does the gradient vanish?
*✍️ By hand · ★☆☆*

(a) Show that $\max_z\sigma'(z) = 0.25$. (b) Compute the bound $0.25^5$ for 5 sigmoid layers. (c) What is the smallest depth $L$ for which the bound
$0.25^L$ drops below $10^{-6}$? (d) What is the corresponding factor for ReLU units that are active?

In [ ]:
bound_5 = None
L_min = None

check('0.25^5', bound_5, 0.25 ** 5)
check('smallest L', L_min, next(L for L in range(1, 100) if 0.25 ** L < 1e-6))

<details>
<summary><b>💡 Hint</b></summary>

$\sigma'(z) = \sigma(1-\sigma)$ is a parabola in $\sigma\in(0,1)$. For (c) solve $L > 6/\log_{10}4$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $s(1-s)$ is maximal at $s = \tfrac12$, i.e. $z = 0$, value $\tfrac14$.
(b) $0.25^5 = 1/1024\approx\mathbf{0.000977}$.
(c) $L > 6/\log_{10}4 = 6/0.602 = 9.97$, so $L = \mathbf{10}$ ($0.25^{10}\approx9.5\times10^{-7}$).
(d) Active ReLU: derivative 1, so the activation factor is $1^L = 1$; only the weights scale the signal.

```python
bound_5 = 0.25 ** 5
L_min = 10
```

</details>

### Exercise 10 · Terminal velocity and standard error
*✍️ By hand · ★★☆*

(a) Momentum with $\eta = 0.01$, $\alpha = 0.9$ and a constant gradient $\mathbf{g} = (2, -1)$: derive and compute the terminal velocity $\mathbf{v}_\infty$.
(b) Per-example gradients have standard deviation $\sigma = 4$. What is the standard error of the mini-batch mean for $B = 16$? Which $B$ gives $SE = 0.25$?

In [ ]:
v_inf = None
se_16 = None
B_needed = None

_v = np.zeros(2)
for _ in range(2000):
    _v = 0.9 * _v - 0.01 * np.array([2, -1.])
check('terminal velocity (simulated)', v_inf, _v)
check('SE(B=16)', se_16, 4 / np.sqrt(16)); check('B for SE=0.25', B_needed, (4 / 0.25) ** 2)

<details>
<summary><b>💡 Hint</b></summary>

Fixed point of $\mathbf{v} = \alpha\mathbf{v} - \eta\mathbf{g}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $\mathbf{v}_\infty = -\eta\mathbf{g}/(1-\alpha) = -0.01(2,-1)/0.1 = \mathbf{(-0.2, 0.1)}$, ten times the plain GD step $-\eta\mathbf{g} = (-0.02, 0.01)$.
(b) $SE = 4/\sqrt{16} = \mathbf{1}$. For $SE = 0.25$: $\sqrt B = 16$, $B = \mathbf{256}$: 16× the computation for 4× less noise.

```python
v_inf = np.array([-0.2, 0.1])
se_16 = 1.0
B_needed = 256
```

</details>

### Exercise 11 · Gradient clipping by hand
*✍️ By hand · ★☆☆*

Clip with threshold $v = 1$: (a) $\mathbf{g} = (3, 4)$, (b) $\mathbf{g} = (0.3, 0.4)$. Then (c) show that the clipped vector always has the same direction as $\mathbf{g}$.

In [ ]:
g_clip_a = None
g_clip_b = None

check('clip (3,4)', g_clip_a, np.array([3, 4.]) / 5)
check('clip (0.3,0.4)', g_clip_b, [0.3, 0.4])

<details>
<summary><b>💡 Hint</b></summary>

$\lVert(3,4)\rVert = 5$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $\lVert\mathbf{g}\rVert = 5 > 1$, so $\mathbf{g}\cdot1/5 = \mathbf{(0.6, 0.8)}$. (b) $\lVert\mathbf{g}\rVert = 0.5\le1$: unchanged, $(0.3, 0.4)$.
(c) The clipped vector is $c\,\mathbf{g}$ with $c = v/\lVert\mathbf{g}\rVert>0$: a positive multiple, so the same direction, length exactly $v$.

```python
g_clip_a = np.array([0.6, 0.8])
g_clip_b = np.array([0.3, 0.4])
```

</details>

### Exercise 12 · Softmax + cross-entropy: the gradient is o − t
*✍️ By hand · ★★★*

(a) For $o_k = e^{z_k}/\sum_je^{z_j}$ and $\mathcal L = -\sum_kt_k\ln o_k$ with one-hot $\mathbf{t}$, show $\partial\mathcal L/\partial z_k = o_k - t_k$.
(b) Evaluate for logits $\mathbf{z} = (2, 1, 0)$ and true class 1 (the first one): the probabilities, the loss and the gradient.

In [ ]:
o_soft = None
loss_ce = None
grad_z = None

_z = np.array([2, 1, 0.]); _t = np.array([1, 0, 0.])
check('softmax', o_soft, softmax(_z), tol=1e-3)
check('loss', loss_ce, -log_softmax(_z)[0], tol=1e-3)
check('gradient', grad_z, softmax(_z) - _t, tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

$\ln o_k = z_k - \ln\sum_je^{z_j}$, so $\mathcal L = -z_c + \ln\sum_je^{z_j}$ for the true class $c$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $\mathcal L = -z_c + \ln\sum_je^{z_j}$, so $\partial\mathcal L/\partial z_k = -[k=c] + e^{z_k}/\sum_je^{z_j} = o_k - t_k$. The same simple form as logistic regression's $\sigma_n - y_n$.
(b) $e^{2}, e^{1}, e^{0} = 7.389, 2.718, 1$; sum $11.107$, so $\mathbf{o} = (0.665, 0.245, 0.090)$, $\mathcal L = -\ln0.665 = \mathbf{0.408}$,
gradient $(-0.335, 0.245, 0.090)$. It pushes $z_1$ up and the others down, proportionally to their probability.

```python
o_soft = np.exp([2, 1, 0]) / np.exp([2, 1, 0]).sum()
loss_ce = -np.log(o_soft[0])
grad_z = o_soft - np.array([1, 0, 0])
```

</details>

### Exercise 13 · Counting parameters of an MLP
*✍️ By hand · ★☆☆*

How many trainable parameters does an MLP 784 → 128 → 64 → 10 (fully connected, with biases) have? And the self-study CNN:
`Conv2d(1, 15, kernel_size=5)` followed by `Linear(15*24*24, 10)`? (A conv layer with $C_{in}$ input channels, $C_{out}$ filters of size $k\times k$ has $C_{out}(C_{in}k^2+1)$ parameters.)
Why is the feature map $24\times24$?

In [ ]:
n_mlp = None
n_cnn = None

check('MLP', n_mlp, sum(a * b + b for a, b in [(784, 128), (128, 64), (64, 10)]))
check('CNN', n_cnn, 15 * (1 * 25 + 1) + 15 * 24 * 24 * 10 + 10)

<details>
<summary><b>💡 Hint</b></summary>

Each dense layer $a\to b$ has $ab + b$ parameters.

</details>

<details>
<summary><b>✅ Solution</b></summary>

MLP: $784\cdot128+128 = 100\,480$, $128\cdot64+64 = 8\,256$, $64\cdot10+10 = 650$; total $\mathbf{109\,386}$.
CNN: conv $15(25+1) = 390$; dense $8640\cdot10 + 10 = 86\,410$; total $\mathbf{86\,800}$. A $5\times5$ filter without padding fits $28-5+1 = 24$ positions per axis,
hence `xb.view(-1, 15*24*24)` before the linear layer. Note how few parameters the convolution itself needs (weight sharing).

```python
n_mlp = 109386
n_cnn = 390 + 86410
```

</details>

## Part C · Code from scratch

### Exercise 14 · Activations and their derivatives
*💻 Code · ★☆☆*

Implement `sigmoid, d_sigmoid, tanh, d_tanh, relu, d_relu` (derivatives as functions of $z$; use $d\,\text{ReLU}(0) := 0$).
Check every derivative against central finite differences on a grid that avoids $z = 0$, and plot the three derivatives.

In [ ]:
def sigmoid(z):   return None
def d_sigmoid(z): return None
def tanh(z):      return None
def d_tanh(z):    return None
def relu(z):      return None
def d_relu(z):    return None

z_grid = np.linspace(-5, 5, 400) + 1e-3     # shifted so that 0 is not on the grid

_h = 1e-6
for _f, _df, _name in [(sigmoid, d_sigmoid, 'sigmoid'), (tanh, d_tanh, 'tanh'), (relu, d_relu, 'relu')]:
    if _f(z_grid) is None or _df(z_grid) is None:
        check(_name, None, 0); continue
    check(_name + ' values', _f(z_grid), {'sigmoid': expit, 'tanh': np.tanh, 'relu': lambda z: np.maximum(z, 0)}[_name](z_grid))
    check(_name + ' derivative', _df(z_grid), (_f(z_grid + _h) - _f(z_grid - _h)) / (2 * _h), tol=1e-5)

<details>
<summary><b>💡 Hint</b></summary>

Use $\sigma' = \sigma(1-\sigma)$ and $\tanh' = 1-\tanh^2$. `scipy.special.expit` is a numerically safe sigmoid.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The plot shows $\sigma'\le0.25$, $\tanh'\le1$ (both vanish for large $|z|$: saturation), and the ReLU derivative is a step: 0 or 1.

```python
def sigmoid(z):   return expit(z)
def d_sigmoid(z): return expit(z) * (1 - expit(z))
def tanh(z):      return np.tanh(z)
def d_tanh(z):    return 1 - np.tanh(z) ** 2
def relu(z):      return np.maximum(z, 0)
def d_relu(z):    return (z > 0).astype(float)

z_grid = np.linspace(-5, 5, 400) + 1e-3
plt.figure(figsize=(5, 3))
for df, name in [(d_sigmoid, "sigmoid'"), (d_tanh, "tanh'"), (d_relu, "ReLU'")]:
    plt.plot(z_grid, df(z_grid), label=name)
plt.legend(); plt.show()
```

</details>

### Exercise 15 · Backprop the slide graph in code
*💻 Code · ★★☆*

Write `graph_forward_backward(x1, w1, x2, w2, o)` that returns `err` and a dict of gradients `{'x1','w1','x2','w2'}`, computed node by node
(store the forward values, then walk the graph backwards, summing at the branch $y_3$). Run it on the slide values and on random inputs where $y_3<0$
(what happens to the ReLU branch there?).

In [ ]:
def graph_forward_backward(x1, w1, x2, w2, o=1.0):
    # TODO
    return None, None

err_slide, grads_slide = graph_forward_backward(1.0, 2.0, 3.0, 0.5)

def _e(p):
    a = p[0] * p[1] + p[2] * p[3]
    return (1 - expit(a) * max(a, 0)) ** 2
for _p in ([1., 2., 3., 0.5], [0.3, -1.2, 0.7, 0.4]):
    _pa = np.array(_p); _num = num_grad(lambda: _e(_pa), _pa)
    _err, _g = graph_forward_backward(*_p)
    check(f'gradients at {_p}', None if _g is None else [_g['x1'], _g['w1'], _g['x2'], _g['w2']], _num, tol=1e-5)

<details>
<summary><b>💡 Hint</b></summary>

Local derivatives: product node passes `upstream * other_input`, add node passes `upstream` to both inputs, sigmoid multiplies by `s*(1-s)`, ReLU by `y3 > 0`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

For $y_3<0$ the ReLU outputs 0 and has derivative 0, so $y_6 = 0$, $\text{err} = 1$, and both branches give zero gradient: $\partial\text{err}/\partial y_4 = \partial\text{err}/\partial y_6\cdot y_5 = 0$.
Every parameter gradient is 0: the unit is "dead" for that input. This is the same mechanism as in Exercise 2.

```python
def graph_forward_backward(x1, w1, x2, w2, o=1.0):
    y1, y2 = x1 * w1, x2 * w2
    y3 = y1 + y2
    y4 = expit(y3); y5 = max(y3, 0.0)
    y6 = y4 * y5
    err = (o - y6) ** 2
    d6 = -2 * (o - y6)
    d4, d5 = d6 * y5, d6 * y4
    d3 = d4 * y4 * (1 - y4) + d5 * (1.0 if y3 > 0 else 0.0)
    d1 = d2 = d3
    return err, {'x1': d1 * w1, 'w1': d1 * x1, 'x2': d2 * w2, 'w2': d2 * x2}

err_slide, grads_slide = graph_forward_backward(1.0, 2.0, 3.0, 0.5)
print(round(err_slide, 4), {k: round(v, 3) for k, v in grads_slide.items()})
```

</details>

### Exercise 16 · A numerically stable softmax and cross-entropy
*💻 Code · ★★☆*

Implement `stable_softmax(Z)` and `log_softmax_rows(Z)` for a matrix of logits (one row per sample) using the log-sum-exp trick, and
`ce_from_logits(Z, y)` = mean of $-\log o_{n,y_n}$. Test on logits as large as 1000, where the naive formula returns `nan`.

In [ ]:
Z_big = np.array([[1000., 999., 0.], [-1000., -1001., -999.], [1., 2., 3.]])
y_big = np.array([0, 2, 1])

def stable_softmax(Z):
    return None
def log_softmax_rows(Z):
    return None
def ce_from_logits(Z, y):
    return None

check('softmax', stable_softmax(Z_big), softmax(Z_big, axis=1))
check('log-softmax', log_softmax_rows(Z_big), log_softmax(Z_big, axis=1))
check('cross-entropy', ce_from_logits(Z_big, y_big), -np.mean(log_softmax(Z_big, axis=1)[np.arange(3), y_big]))

<details>
<summary><b>💡 Hint</b></summary>

$\log\sum_je^{z_j} = m + \log\sum_je^{z_j - m}$ with $m = \max_jz_j$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Subtracting the row maximum leaves softmax unchanged but makes all exponents $\le0$, so nothing overflows; computing `log_softmax` directly also avoids
$\log(0) = -\infty$ for tiny probabilities. This is what `nn.CrossEntropyLoss` does internally.

```python
Z_big = np.array([[1000., 999., 0.], [-1000., -1001., -999.], [1., 2., 3.]])
y_big = np.array([0, 2, 1])

def log_softmax_rows(Z):
    m = Z.max(1, keepdims=True)
    return Z - m - np.log(np.exp(Z - m).sum(1, keepdims=True))
def stable_softmax(Z):
    return np.exp(log_softmax_rows(Z))
def ce_from_logits(Z, y):
    return -np.mean(log_softmax_rows(Z)[np.arange(len(y)), y])

with np.errstate(over='ignore', invalid='ignore'):
    print('naive:', (np.exp(Z_big) / np.exp(Z_big).sum(1, keepdims=True))[0])
```

</details>

### Exercise 17 · A two-layer MLP from scratch
*💻 Code · ★★★*

Build an MLP $2\to H\to1$ with $\tanh$ hidden units and a sigmoid output, trained with mean binary cross-entropy on `make_moons`:
$$\mathbf{Z}_1 = \mathbf{X}\mathbf{W}_1 + \mathbf{b}_1,\ \mathbf{H} = \tanh(\mathbf{Z}_1),\ \mathbf{a} = \mathbf{H}\mathbf{W}_2+b_2,\ \mathbf{o} = \sigma(\mathbf{a}).$$

1. Implement `forward`, `loss` and `backward` (returning a dict of gradients with the same keys as the parameters).
2. Gradient-check `backward` with `num_grad` (store the max abs error in `mlp_grad_err`).
3. Train with full-batch gradient descent (lr 1.0, 3000 steps, $H = 16$) and store the training accuracy in `mlp_acc`.

In [ ]:
X_moon, y_moon = make_moons(300, noise=0.2, random_state=0)

def init_params(H, seed=0):
    r = np.random.default_rng(seed)
    return {'W1': r.normal(0, 1, (2, H)), 'b1': np.zeros(H), 'W2': r.normal(0, 1 / np.sqrt(H), (H, 1)), 'b2': np.zeros(1)}

def forward(p, X):
    # TODO: return o (shape N) and a cache for backward
    return None, None

def loss(p, X, y):
    return None

def backward(p, X, y):
    # TODO: dict of gradients, same keys and shapes as p
    return None

params = init_params(16)
mlp_grad_err = None
mlp_acc = None

check('gradient check < 1e-6', None if mlp_grad_err is None else mlp_grad_err < 1e-6, True)
check('training accuracy > 0.95', None if mlp_acc is None else mlp_acc > 0.95, True)

<details>
<summary><b>💡 Hint 1</b></summary>

With sigmoid + BCE the output delta is simply $\boldsymbol\delta_a = (\mathbf{o}-\mathbf{y})/N$ (cf. Exercise 12).

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Then $\nabla\mathbf{W}_2 = \mathbf{H}^T\boldsymbol\delta_a$, $\boldsymbol\delta_{Z_1} = (\boldsymbol\delta_a\mathbf{W}_2^T)\odot(1-\mathbf{H}^2)$, $\nabla\mathbf{W}_1 = \mathbf{X}^T\boldsymbol\delta_{Z_1}$; biases sum over rows.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The finite-difference error is about $10^{-10}$, so backprop is right; GD reaches about 98 % training accuracy with a curved boundary, which a single linear layer (or the
1-unit bottleneck of Exercise 2) cannot represent. This is precisely what `loss.backward()` + `optimizer.step()` automate.

```python
X_moon, y_moon = make_moons(300, noise=0.2, random_state=0)

def init_params(H, seed=0):
    r = np.random.default_rng(seed)
    return {'W1': r.normal(0, 1, (2, H)), 'b1': np.zeros(H), 'W2': r.normal(0, 1 / np.sqrt(H), (H, 1)), 'b2': np.zeros(1)}

def forward(p, X):
    Z1 = X @ p['W1'] + p['b1']
    H = np.tanh(Z1)
    o = expit(H @ p['W2'] + p['b2'])[:, 0]
    return o, H

def loss(p, X, y):
    o, _ = forward(p, X)
    o = np.clip(o, 1e-12, 1 - 1e-12)
    return -np.mean(y * np.log(o) + (1 - y) * np.log(1 - o))

def backward(p, X, y):
    o, H = forward(p, X)
    da = ((o - y) / len(y))[:, None]
    g = {'W2': H.T @ da, 'b2': da.sum(0)}
    dZ1 = (da @ p['W2'].T) * (1 - H ** 2)
    g['W1'] = X.T @ dZ1
    g['b1'] = dZ1.sum(0)
    return g

params = init_params(16)
g_an = backward(params, X_moon, y_moon)
mlp_grad_err = max(np.abs(num_grad(lambda: loss(params, X_moon, y_moon), params[k]) - g_an[k]).max() for k in params)
for _ in range(3000):
    g = backward(params, X_moon, y_moon)
    for k in params:
        params[k] -= 1.0 * g[k]
mlp_acc = np.mean((forward(params, X_moon)[0] > 0.5) == y_moon)
print('grad err', mlp_grad_err, ' acc', mlp_acc)
```

</details>

### Exercise 18 · Measure the vanishing gradient
*💻 Code · ★★☆*

Build a deep stack of $L = 10$ square layers (width 32, no biases) $\mathbf{h}^{(l)} = f(\mathbf{h}^{(l-1)}\mathbf{W}_l)$ on 64 random inputs, with the loss
$\mathcal L = \text{mean}(\text{sum of the last layer})$. Backpropagate by hand and record $\lVert\partial\mathcal L/\partial\mathbf{W}_l\rVert$ for every layer.
Do it for $f = $ sigmoid (init std $1/\sqrt{32}$) and $f = $ ReLU (He init std $\sqrt{2/32}$), and store the ratios
first-layer norm / last-layer norm in `ratio_sigmoid` and `ratio_relu`.

In [ ]:
def layer_grad_norms(act, L=10, width=32, seed=0):
    # TODO: return a list of L gradient norms, layer 1 first
    return None

ratio_sigmoid = None
ratio_relu = None

if ratio_sigmoid is not None:
    check('sigmoid: first layer gets < 1e-4 of the last layer gradient', ratio_sigmoid < 1e-4, True)
    check('ReLU keeps the same order of magnitude (> 0.05)', ratio_relu > 0.05, True)
else:
    check('sigmoid: first layer gets < 1e-4 of the last layer gradient', None, True)

<details>
<summary><b>💡 Hint</b></summary>

Backward step per layer: $\boldsymbol\delta_z = \boldsymbol\delta_h\odot f'(\mathbf{z})$, $\nabla\mathbf{W}_l = \mathbf{h}_{l-1}^T\boldsymbol\delta_z$, $\boldsymbol\delta_{h,l-1} = \boldsymbol\delta_z\mathbf{W}_l^T$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With sigmoids the first layer receives roughly $10^{-7}$ of the last layer's gradient (each layer contributes a factor $\le0.25$, times the weights),
while ReLU with He initialisation keeps the norms of the same order. Plot `np.log10(norms)` against the layer index to see the straight line of exponential decay.

```python
def layer_grad_norms(act, L=10, width=32, seed=0):
    r = np.random.default_rng(seed)
    X = r.normal(size=(64, width))
    std = np.sqrt(2 / width) if act == 'relu' else np.sqrt(1 / width)
    Ws = [r.normal(0, std, (width, width)) for _ in range(L)]
    f = (lambda z: np.maximum(z, 0)) if act == 'relu' else expit
    df = (lambda z: (z > 0).astype(float)) if act == 'relu' else (lambda z: expit(z) * (1 - expit(z)))
    hs, zs = [X], []
    for W in Ws:
        zs.append(hs[-1] @ W); hs.append(f(zs[-1]))
    d = np.ones_like(hs[-1]) / len(X)
    norms = []
    for l in reversed(range(L)):
        dz = d * df(zs[l])
        norms.append(np.linalg.norm(hs[l].T @ dz))
        d = dz @ Ws[l].T
    return norms[::-1]

ns, nr = layer_grad_norms('sigmoid'), layer_grad_norms('relu')
ratio_sigmoid, ratio_relu = ns[0] / ns[-1], nr[0] / nr[-1]
print('sigmoid ratio %.2e, relu ratio %.2f' % (ratio_sigmoid, ratio_relu))
plt.figure(figsize=(5, 3)); plt.semilogy(range(1, 11), ns, 'o-', label='sigmoid'); plt.semilogy(range(1, 11), nr, 's-', label='ReLU')
plt.xlabel('layer'); plt.ylabel('||dL/dW||'); plt.legend(); plt.show()
```

</details>

### Exercise 19 · Gradient descent vs. momentum vs. Nesterov in a narrow valley
*💻 Code · ★★☆*

Minimise $f(\mathbf{w}) = \tfrac12\mathbf{w}^T\mathbf{A}\mathbf{w}$ with $\mathbf{A} = \operatorname{diag}(1, 50)$ from $\mathbf{w}_0 = (1, 1)$ for 100 steps with $\eta = 0.019$:
plain GD, classical momentum ($\alpha = 0.9$) and Nesterov ($\alpha = 0.9$). Store the final losses in `loss_gd`, `loss_mom`, `loss_nag`.
Why can't plain GD simply use a larger $\eta$?

In [ ]:
A_valley = np.diag([1., 50.])
f_valley = lambda w: 0.5 * w @ A_valley @ w
grad_valley = lambda w: A_valley @ w
loss_gd = None
loss_mom = None
loss_nag = None

if loss_gd is not None:
    check('momentum beats GD by >100x', loss_mom < loss_gd / 100, True)
    check('Nesterov beats GD by >100x', loss_nag < loss_gd / 100, True)
else:
    check('momentum beats GD by >100x', None, True)

<details>
<summary><b>💡 Hint</b></summary>

GD on a quadratic is stable only for $\eta<2/\lambda_{\max} = 0.04$; with $\eta = 0.019$ the flat direction ($\lambda = 1$) shrinks by only $0.981$ per step.

</details>

<details>
<summary><b>✅ Solution</b></summary>

GD is limited by the steep direction ($\eta<2/50$), so the flat direction converges painfully slowly ($0.981^{100}\approx0.15$, loss about $10^{-2}$).
Momentum accumulates velocity along the flat direction and damps the oscillation across the valley: loss about $10^{-5}$ or lower; Nesterov is similar or slightly better.

```python
A_valley = np.diag([1., 50.])
f_valley = lambda w: 0.5 * w @ A_valley @ w
grad_valley = lambda w: A_valley @ w
eta, alpha = 0.019, 0.9
w = np.array([1., 1.])
for _ in range(100):
    w = w - eta * grad_valley(w)
loss_gd = f_valley(w)
w, v = np.array([1., 1.]), np.zeros(2)
for _ in range(100):
    v = alpha * v - eta * grad_valley(w); w = w + v
loss_mom = f_valley(w)
w, v = np.array([1., 1.]), np.zeros(2)
for _ in range(100):
    v = alpha * v - eta * grad_valley(w + alpha * v); w = w + v
loss_nag = f_valley(w)
print('GD %.2e  momentum %.2e  Nesterov %.2e' % (loss_gd, loss_mom, loss_nag))
```

</details>

### Exercise 20 · Mini-batch noise scales like 1/√B
*💻 Code · ★★☆*

For linear regression with squared loss on the synthetic data below, compute the **full** gradient at a fixed $\mathbf{w}$, then draw 2000 random
mini-batches (with replacement) of size $B\in\{10, 40, 160\}$ and measure the standard deviation of the first gradient component.
Store `se_by_B` (dict) and the ratio `se_by_B[10] / se_by_B[40]`. What does theory predict?

In [ ]:
r = np.random.default_rng(8)
X_sgd = r.normal(size=(10000, 3)); y_sgd = X_sgd @ np.array([1., -2., 0.5]) + r.normal(size=10000)
w_fixed = np.zeros(3)
per_example_grads = None   # (N, 3): gradient of (x^T w - y)^2 for each example
se_by_B = None
ratio_10_40 = None

if ratio_10_40 is not None:
    check('ratio close to sqrt(4) = 2', abs(ratio_10_40 - 2) < 0.2, True)
    _sigma = (2 * (X_sgd @ w_fixed - y_sgd)[:, None] * X_sgd)[:, 0].std()
    check('SE(B=160) matches sigma/sqrt(160)', se_by_B[160], _sigma / np.sqrt(160), tol=0.02)
else:
    check('ratio close to sqrt(4) = 2', None, True)

<details>
<summary><b>💡 Hint</b></summary>

Per-example gradient: $2(\mathbf{x}_n^T\mathbf{w}-y_n)\mathbf{x}_n$. Sample indices with `r.integers(0, N, size=(2000, B))` and average along axis 1.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$SE = \sigma/\sqrt B$, so quadrupling $B$ halves the noise: the measured ratio is about 2, and $B=160$ is only $4\times$ less noisy than $B=10$ at $16\times$ the cost.

```python
r = np.random.default_rng(8)
X_sgd = r.normal(size=(10000, 3)); y_sgd = X_sgd @ np.array([1., -2., 0.5]) + r.normal(size=10000)
w_fixed = np.zeros(3)
per_example_grads = 2 * (X_sgd @ w_fixed - y_sgd)[:, None] * X_sgd
se_by_B = {}
for B in (10, 40, 160):
    idx = r.integers(0, len(y_sgd), size=(2000, B))
    se_by_B[B] = per_example_grads[idx, 0].mean(1).std()
ratio_10_40 = se_by_B[10] / se_by_B[40]
print({B: round(v, 3) for B, v in se_by_B.items()}, 'ratio', round(ratio_10_40, 3))
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Lecture Notes 1-12](../Lecture%20Notes%201-12.md).*